<h1>MonReader: Single Frame Modelling (SFM)

Treat all images as their own row regardless of if they are frames part of the same clip. Sequential frames of the same clip are similar but slightly different so each image should in theory contribute information to model training.

In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
from pathlib import Path
PROJ_ROOT = Path().resolve().parents[0]
DATA_DIR = PROJ_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
TRAIN_DATA_DIR = RAW_DATA_DIR / "training"
TEST_DATA_DIR = RAW_DATA_DIR / "testing"

<h3>Load All Training/Testing Data Into Single Datasets

In [3]:
import cv2

ALLOWED_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.gif'}

def load_images_to_df(image_dir, label=None, label_from_subdir=True, split=None):
    """
    Load images into a DataFrame using cv2.imread.

    Images are converted from OpenCV's BGR format to RGB.
    """
    records = []
    image_dir = Path(image_dir)

    if not image_dir.is_dir():
        return pd.DataFrame(records)

    def _load_from_dir(directory, lbl):
        for root_dir, _, files in os.walk(directory):
            for fname in sorted(files):
                path = os.path.join(root_dir, fname)

                if Path(fname).suffix.lower() not in ALLOWED_EXTS:
                    continue

                image = cv2.imread(path, cv2.IMREAD_COLOR)
                if image is None:
                    continue

                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                records.append({
                    'image': image,
                    'label': lbl,
                    'split': split
                })

    if label_from_subdir:
        subdirs = [
            d for d in sorted(os.listdir(image_dir))
            if (image_dir / d).is_dir()
        ]

        if subdirs:
            for subdir in subdirs:
                _load_from_dir(image_dir / subdir, subdir)
        else:
            _load_from_dir(image_dir, label)
    else:
        _load_from_dir(image_dir, label)

    return pd.DataFrame(records)


In [4]:
raw_training = load_images_to_df(image_dir=TRAIN_DATA_DIR,
                             label=False,
                             label_from_subdir=True,
                             split='training')

In [5]:
raw_testing = load_images_to_df(image_dir=TEST_DATA_DIR,
                             label=False,
                             label_from_subdir=True,
                             split='testing')

In [6]:
raw_training.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 2392 entries, 0 to 2391
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   image   2392 non-null   object
 1   label   2392 non-null   str   
 2   split   2392 non-null   str   
dtypes: object(1), str(2)
memory usage: 13.9 GB


pandas likely underreports the size because image contains NumPy arrays inside an object column. Measure the image data directly:

In [7]:
image_bytes = raw_training["image"].map(lambda image: image.nbytes).sum()
print(f"Image data: {image_bytes / 1024**2:.2f} MB")
print(f"DataFrame reported: {raw_training.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Image data: 14190.82 MB
DataFrame reported: 14191.42 MB


In [8]:
image_bytes = raw_testing["image"].map(lambda image: image.nbytes).sum()
print(f"Image data: {image_bytes / 1024**2:.2f} MB")
print(f"DataFrame reported: {raw_testing.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Image data: 3541.77 MB
DataFrame reported: 3541.92 MB


My biggest concern at the moment is the size of the variables holding the images, as they are high resolution and each variable (train and test) will contain 1000s of rows. In total they are taking up ~17-18GB in memory

In [9]:
raw_training.head(5)

,image,label,split
0,"[[[145, 137, 118], [145, 137, 118], [145, 137,...",flip,training
1,"[[[145, 137, 118], [145, 137, 118], [145, 137,...",flip,training
2,"[[[144, 140, 113], [143, 139, 112], [142, 138,...",flip,training
3,"[[[144, 140, 113], [143, 139, 112], [142, 138,...",flip,training
4,"[[[149, 145, 118], [142, 138, 111], [135, 131,...",flip,training


In [10]:
print(f"Shape: {raw_training.shape}")
print(f"Image data-type: {type(raw_training["image"][0])}, Shape: {raw_training["image"][0].shape}")
print(f"Image format: {["RGB" if raw_training["image"][0].shape[2] == 3 else "Non-RGB"]}")

Shape: (2392, 3)
Image data-type: <class 'numpy.ndarray'>, Shape: (1920, 1080, 3)
Image format: ['RGB']


We'll need to convert the RGB images here into grayscale like we did in the last notebook. store the new variables, then wipe the raw varaibles from memory for efficiency i.e. Garbage collection.

e.g.

import gc

del x

gc.collect()

In [11]:
import gc
del raw_training
del raw_testing
gc.collect()

0

In [12]:
def convert_images_to_grayscale(df):
    """
    Return a copy of df with images converted from RGB to grayscale.
    """
    grayscale_df = df.copy()
    grayscale_df["image"] = grayscale_df["image"].map(
        lambda image: cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    )
    return grayscale_df

Import raw data -> Use the grayscale conversion function - > Delete the raw data set from memory for efficiency:

In [13]:
raw_training = load_images_to_df(image_dir=TRAIN_DATA_DIR,
                             label=False,
                             label_from_subdir=True,
                             split='training')

gray_training = convert_images_to_grayscale(raw_training)

del raw_training

gray_training.head(5)

,image,label,split
0,"[[137, 137, 137, 137, 137, 137, 137, 137, 137,...",flip,training
1,"[[137, 137, 137, 137, 137, 137, 137, 137, 137,...",flip,training
2,"[[138, 137, 136, 136, 136, 136, 137, 137, 137,...",flip,training
3,"[[138, 137, 136, 136, 136, 136, 137, 137, 137,...",flip,training
4,"[[143, 136, 129, 129, 134, 138, 137, 135, 137,...",flip,training


In [16]:
print(f"Shape: {gray_training.shape}")
print(f"Image data-type: {type(gray_training["image"][0])}, Shape: {gray_training["image"][0].shape}")
image_bytes = gray_training["image"].map(lambda image: image.nbytes).sum()
print(f"Image data: {image_bytes / 1024**2:.2f} MB")
print(f"DataFrame reported: {gray_training.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Shape: (2392, 3)
Image data-type: <class 'numpy.ndarray'>, Shape: (1920, 1080)
Image data: 4730.27 MB
DataFrame reported: 4730.84 MB


Training set was ~14GB before, now it's ~5GB. Massive improvement

In [17]:
raw_testing = load_images_to_df(image_dir=TEST_DATA_DIR,
                             label=False,
                             label_from_subdir=True,
                             split='testing')

gray_testing = convert_images_to_grayscale(raw_testing)

del raw_testing

gray_testing.head(5)

print(f"Shape: {gray_testing.shape}")
print(f"Image data-type: {type(gray_testing["image"][0])}, Shape: {gray_testing["image"][0].shape}")
image_bytes = gray_testing["image"].map(lambda image: image.nbytes).sum()
print(f"Image data: {image_bytes / 1024**2:.2f} MB")
print(f"DataFrame reported: {gray_testing.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Shape: (597, 3)
Image data-type: <class 'numpy.ndarray'>, Shape: (1920, 1080)
Image data: 1180.59 MB
DataFrame reported: 1180.73 MB


1) Split a validation set from train set.
2) Resize images to a smaller size + add channel dimension (1 = grayscale, 3 = RGB).
3) Normalise pixel values + make sure datatype = float32.
4) Encode the labels

In [18]:
def resize_image(image, scale_factor=0.5):
    """
    Resize a grayscale image by a fixed scale factor while preserving aspect ratio.
    """
    height, width = image.shape[:2]
    new_width = max(1, int(width * scale_factor))
    new_height = max(1, int(height * scale_factor))

    return cv2.resize(image, (new_width, new_height), interpolation=cv2.INTER_AREA)
